# Đánh giá ABSA chỉ dùng train

Theo yêu cầu mới nhất, chỉ dùng `train.csv`; không đọc hoặc dự đoán từ file blind. Notebook bật bộ chặn mở file blind ngay khi bắt đầu. Điểm bên dưới là validation từ train, không phải điểm công khai. Validation đã được xem trong nhiều lượt phát triển.

Mặc định notebook tái tính meta và chỉ số từ đầu ra encoder trên validation đã lưu. Nó không huấn luyện lại encoder. Cách chạy encoder thực sự trên validation có trong TRAIN_ONLY_README.md.


In [1]:
import os, sys, json, time
from pathlib import Path
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
import train_only
import numpy as np, pandas as pd
from IPython.display import display
from phobert_absa import split_indices
from stacked_absa import file_hash
from evaluate_train_only import evaluate
TRAIN = Path(os.environ.get('ABSA_TRAIN_CSV', '../data/train.csv'))
if (ROOT / 'models/stacked/config.json').exists():
    MODEL = ROOT / 'models/stacked'
else:
    names = ['stacked_outer80_results', 'stacked_outer80_visobert_train_only_results', 'stacked_nli_head_train_only_results']
    available = [ROOT / n for n in names if (ROOT / n / 'metrics.json').exists()]
    MODEL = max(available, key=lambda p: json.loads((p/'metrics.json').read_text())['ABSA_F1'])
OUT = ROOT / 'train_only_notebook_review'
print('Chỉ dùng train.csv; bộ chặn file blind đã bật.')


/workspace/work/deep_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chỉ dùng train.csv; bộ chặn file blind đã bật.


In [2]:
cfg = json.loads((MODEL / 'config.json').read_text())
assert file_hash(TRAIN) == cfg['train_sha256']
train = pd.read_csv(TRAIN)
fi, tu, va = split_indices(train, cfg['seed'])
assert len(train) == 4688 and len(va) == 938
display(pd.DataFrame({'phần': ['fit ban đầu', 'tune ban đầu', 'validation giữ riêng'], 'số mẫu': [len(fi), len(tu), len(va)]}))
print('Các encoder refit outer80 chỉ học fit+tune; validation vẫn được giữ riêng.')


                   phần  số mẫu
0           fit ban đầu    2812
1          tune ban đầu     938
2  validation giữ riêng     938

Các encoder refit outer80 chỉ học fit+tune; validation vẫn được giữ riêng.


In [3]:
started = time.perf_counter()
report = evaluate(MODEL, TRAIN, OUT, fresh=False, threads=2, batch_size=8)
seconds = time.perf_counter() - started
print('Thời gian tái tính từ đầu ra validation đã lưu:', round(seconds, 3), 'giây')
print('Chưa tính thời gian huấn luyện hay chạy lại encoder trong con số này.')
print('ABSA-F1 validation:', report['ABSA_F1'])
print('Mục tiêu 0.96 đã đạt:', report['ABSA_F1'] >= 0.96)


Train validation source ready: 1 / 5


Train validation source ready: 2 / 5


Train validation source ready: 3 / 5


Train validation source ready: 4 / 5


Train validation source ready: 5 / 5


Số mẫu: 938
ABSA-F1 (điểm xếp hạng) = 0.884   CI95% [0.870, 0.898]
F1 theo cực: khen 0.931 | chê 0.837 | trung tính (phụ) 0.000 | phát hiện khía cạnh (phụ) 0.927
Nhãn yếu nhất: tien_nghi_che F1 = 0.785; độ lệch chuẩn F1 14 nhãn = 0.064
Precision TB = 0.882 | Recall TB = 0.888 | Accuracy TB = 0.963 (14 nhãn)
  khen: P 0.928 R 0.936 | chê: P 0.837 R 0.840
kappa TB = 0.860 | MCC TB = 0.860 | Balanced acc TB = 0.932
F1 vi mô: khía cạnh = 0.934 | (khía cạnh, cực) = 0.903 | khớp đúng cả đánh giá = 0.638
Mốc lớp đa số =   nan; cải thiện tương đối =   nan

Khía cạnh      Nhắc F1 | Khen F1     P     R     κ     n | Chê F1      P     R     κ     n | Trung tính F1  n
nhan_vien      0.958  | 0.963 0.967 0.960 0.923   495 | 0.825  0.814 0.836 0.802   110 | 0.000       0
sach_se        0.971  | 0.982 0.987 0.977 0.973   310 | 0.861  0.912 0.816 0.850    76 | 0.000       0
vi_tri         0.944  | 0.945 0.967 0.924 0.918   315 | 0.810  0.758 0.870 0.798    54 | 0.000       2
phong          0.935  | 0.

Thời gian tái tính từ đầu ra validation đã lưu: 3.865 giây
Chưa tính thời gian huấn luyện hay chạy lại encoder trong con số này.
ABSA-F1 validation: 0.8842455443513071
Mục tiêu 0.96 đã đạt: False


## Năm ví dụ sai từ phần validation của train

Các ví dụ sau chỉ lấy từ train. Đây là nội dung báo cáo riêng; không đăng đầu ra ô này lên repository công khai. Chúng phản ánh khó khăn còn lại, không được dùng để sửa nhãn blind.


In [ ]:
examples = json.loads((OUT / 'training_validation_errors.json').read_text())
assert len(examples) >= 5
display(pd.DataFrame(examples[:5]))


In [5]:
rows = []
for i, source in enumerate(cfg['sources']):
    folder = MODEL / source['path']
    base = json.loads((folder / 'config.json').read_text())
    rows.append({'encoder': i+1, 'mô hình': base['model_name'], 'kiểu': base.get('architecture','joint_21'), 'refit': base.get('refit_scope','không'), 'mẫu học': base.get('fit_rows',cfg.get('base_fit_rows',2812))})
display(pd.DataFrame(rows))
print('Không công bố điểm công khai từ kết quả validation này.')


   encoder                mô hình             kiểu    refit  mẫu học
0        1  vinai/phobert-base-v2         joint_21  outer80     3750
1        2  vinai/phobert-base-v2         joint_21  outer80     3750
2        3  vinai/phobert-base-v2      aspect_pair    không     2812
3        4        uitnlp/visobert         joint_21    không     2812
4        5  vinai/phobert-base-v2  label_attention  outer80     3750

Không công bố điểm công khai từ kết quả validation này.


Các script huấn luyện, cấu hình, trọng số và báo cáo thời gian nằm cùng gói. Chạy script qua `python train_only.py TEN_SCRIPT ...` để giữ bộ chặn mở file blind. Không chạy các cell suy luận blind của notebook lịch sử.
